In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# 📘 CityAI – Decisions sobre inversions en inmobles

Aquest notebook utilitza un escenari plausible de l'Ajuntament de Barcelona per aplicar l'algorisme de la motxilla en un problema real.

## 🧠 Objectius

Aprofundir en els algorismes vistos a teoria, i veure la seva aplicació en un cas real.

## 🧩 Context

L'Ajuntament de Barcelona disposa de 20 milions d'euros per invertir en projectes immobiliaris de millora urbana. Té una llista de possibles inversions.

|            Projecte         | Cost (M€) | Retorn economic (M€) | Impacte social (sobre 10) |
| ----------------------------|-----------|----------------------|------------------|
| Habitatge social al Raval           | 8  | 10 | 9|
| Reforma mercat de Sant Antoni       | 6  | 7  | 6|
| Centre cívic a Nou Barris           | 4  | 5  | 8|
| Rehabilitació edificis Eixample     | 10 | 12 | 5|
| Oficina d'atenció ciutadana a Sants | 3  | 3  | 7|
| Habitatge sostenible al Poblenou    | 7  | 9  | 8|

Considerarem que el valor de cada inversió és:

valor = 0.8 x retorn_economic + 0.2 x impacte_social


In [6]:
# Per facilitar les proves, es proporcionen les dades inicials aquí mateix  

inversions = [
    ("Habitatge social al Raval", 8, 10, 9), 
    ("Reforma mercat de Sant Antoni", 6, 7, 6),
    ("Centre cívic a Nou Barris", 4, 5, 8),
    ("Rehabilitació edificis Eixample", 10, 12, 5),    
    ("Oficina d'atenció ciutadana a Sants", 3, 3, 7),
    ("Habitatge sostenible al Poblenou", 7, 9, 8)
]
pressupost = 20

## ✍️ Exercici 1: Implementa l'algorisme de la motxilla amb les dades del cas

Tenim un conjunt d'opcions (immobles) cadascuna amb un cost i un benefici esperat (econòmic, social o mixt). L'Ajuntament disposa d'un pressupost màxim i vol maximitzar el benefici total.

Podem fer el paral·lelisme amb la motxilla:
+ Pes => cost d'inversió
+ Valor => tendiment esperat
+ Capacitat màxima => Pressupost total

Adapta el codi de la motxilla amb el cas descrit i amb el valor ponderat descrit més amunt.

Quines són les inversions triades?

In [12]:
from itertools import combinations

def motxilla_fb(
    nombre:int, capacitat:int, pes_cost:list[any]
) -> (int,list[int]):
    """
    Tria la millor combinació d'objectes amb l'algoritme de la motxilla 

    Parametres:
    :param nombre: nombre d'items possibles
    :param capacitat: capacitat de la motxilla
    :param pes_cost:
        llista de tuples del tipus [(pes, cost), (pes, cost), ...]

    Retorn:
        Tupla on el primer element és el cost i el segon element és la
        combinació òptima d'objectes.
    
    idea:
    dp[i][c] representa el millor cost amb els primers i items, amb una motxilla de capacitat c
        per cada nou dp[i][c] = max(cost_agafar, cost_no_agafar)
            - cost_agafar = dp[i-1][c - pes_actual] + cost_item_actual
            - cost_no_agafar = dp[i-1][c]
    """
    
    # init dp
    dp = [[None for _ in range(capacitat + 1)] for _ in range(nombre + 1)]
    for c in range(len(dp[0])):  # valor = 0 si hi ha 0 items
        dp[0][c] = 0

    # dp
    for i in range(1, nombre + 1):
        curr_pes = pes_cost[i-1][0]
        curr_cost = pes_cost[i-1][1]
        for c in range(capacitat + 1):
            if curr_pes > c:
                dp[i][c] = dp[i-1][c]
            else:
                no_agafar = dp[i-1][c]
                agafar = dp[i-1][c - curr_pes] + curr_cost

                dp[i][c] = max(no_agafar, agafar)

    millor_cost = dp[nombre][capacitat]

    # backtrace for millor combinacio
    millor_combinacio = [0 for _ in range(nombre)]
    i = nombre
    c = capacitat
    while i > 0:
        if dp[i][c] == dp[i-1][c]:  # no agafat
            i = i-1
        else:
            millor_combinacio[i-1] = 1
            c = c - pes_cost[i-1][0]
            i = i-1
    
    return int(millor_cost), millor_combinacio

In [11]:
motxilla_fb(6, 20, [
    (8, 0.8 * 10 + 0.2 * 9), (6, 0.8 * 7 + 0.2 * 6), (48, 0.8 * 5 + 0.24 * 8), 
    (10, 0.8 * 12 + 0.2 * 5), (3, 0.8 * 3 + 0.2 * 7), (7, 0.8 * 9 + 0.2 * 8)
])

(23, [0, 0, 0, 1, 1, 1])

Quines han sigut les inversions triades?

## ✍️ Exercici 2 – Discussió i reflexió

Què passaria si es donés més pes a l'impacte social que al retorn econòmic? per exemple 0.3 per retorn econòmic i 0.7 per retorn social.

Retoca l'algorisme o la crida i observa els resultats

In [7]:
pes_cost = [(inversio[1], inversio[2] * 0.3 + inversio[3] * 0.7) for inversio in inversions ]

In [10]:
motxilla_fb(6, 20, pes_cost)

(27, [0, 1, 1, 0, 1, 1])

Quines són ara les inversions triades? compara-les amb les anteriors.

## ✍️ Exercici 3 – Crea funcions auxiliars que 

1. Permetin dir el percentatge assignat a cada concepte (retorn econòmic i impacte social) i facin la crida corresponent a l'algorisme de la motxilla. 
1. Crein un diccionari amb la posició a la llista i el nom del projecte i quan l'algorisme retorni el resultat treguin un llistat dels projectes seleccionats.

In [13]:
def percentatge(ret_eco:float, imp_soc:float, llista:list[any]) -> (list[any]):
    """
    Calcula la combinació de projectes òptima en cost i impacte socioeconòmic. 

    Aquesta funció calcula la millor combinació de projectes d'una llista de
    projectes tot utilitzant l'algoritme de la motxilla, permetent personalitzar
    el balanç entre el pes donat al seu impacte social i el pes donat al seu
    retorn econòmic.

    Parametres:
        :param ret_eco: percentatge del retorn econòmic
        :param imp_soc: percentatge de l'impacte social
        :param llista: llista de tuples del tipus 
            [(projecte, pes, retorn_economic, impacte_social), ...]
    
    Retorn:
        Tupla on el primer element és el cost i el segon element és la
        combinació òptima d'objectes.
    """
    pes_cost = [(inversio[1], inversio[2] * ret_eco + inversio[3] * imp_soc) for inversio in inversions ]
    

    return motxilla_fb(len(llista), 20, pes_cost)

In [14]:
percentatge(0.8, 0.2, inversions)

(25, [0, 1, 1, 0, 1, 1])

In [15]:
def resultats(selec:list[int],inversions:list[any]) -> list[str]:
    """
    Tradueix la combinació de projectes a la llista amb els noms dels projectes.

    Parametres:
        :param selec: 
            llista amb els projectes seleccionats (1) i no seleccionats (0)
        :param inversions: 
            llista de tuples del tipus 
            [(projecte, pes, retorn_economic, impacte_social), ...]

    Retorn:
        Llista amb els noms dels projectes que té la combinació introduïda.
    """

    seleccionats = []
    for i in range(len(selec)):
        if selec[i] == 1:
            seleccionats.append(inversions[i][0])
   
    return seleccionats

In [16]:
resultats([1, 0, 0, 1, 0, 1], inversions)

['Habitatge social al Raval',
 'Rehabilitació edificis Eixample',
 'Habitatge sostenible al Poblenou']